# 01 · MLP V3 (딥러닝 전용 개선기법)

> 목적: 머신러닝(XGB/LGBM)은 비교 기준으로 고정하고, **딥러닝에 아직 안 쓴 기법**으로 MLP를 제대로 끌어올린다.
> 적용 기법:
> - **ResidualMLP + Wide&Deep**: 단순 MLP보다 깊고 표현력 있는 구조 (잔차 연결 + 원본 피처 직접 경로)
> - **WeightedRandomSampler**: SMOTE처럼 가짜를 만들지 않고, 배치마다 소수(Yes)를 더 자주 보여줌
> - **Focal Loss (alpha/gamma를 Optuna로 튜닝)**: 지금까진 고정값만 썼음
> - **PR_AUC 기준 early stopping**: loss가 아니라 목표 지표(PR_AUC)가 가장 좋은 시점을 저장
> - **Optuna**: hidden/blocks/dropout/lr/weight_decay/alpha/gamma 탐색
>
> 기록: results_v3_dl.csv. 비교: MLP_v2 0.198 / XGB_v2_tuned 0.296 (고정 기준).
> 정직한 기대: 0.22~0.26 근처면 성공. 0.296 못 넘어도 그대로 기록.

## 0. 환경 + 데이터
처리 V2 데이터(42피처)를 그대로 쓴다. (전처리는 이미 train 기준 fit이라 누수 없음)

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import average_precision_score, f1_score, fbeta_score, recall_score, precision_score
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from utils import set_seed, compute_metrics, log_result, load_processed
set_seed(42)

PROJECT_ROOT = Path(r"C:\Users\Administrator\Desktop\딥러닝응용\TermProject")
OUT_DIR    = PROJECT_ROOT / "processed_v2"
NB_DIR     = PROJECT_ROOT / "notebooks_v3"
RESULTS_V3 = NB_DIR / "results_v3_dl.csv"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_df, val_df, test_df = load_processed(OUT_DIR)
TARGET = "went_on_backorder"
feature_cols = [c for c in train_df.columns if c != TARGET]
input_dim = len(feature_cols)

X_train = train_df[feature_cols].values.astype("float32")
y_train = train_df[TARGET].values.astype("float32")
X_val   = val_df[feature_cols].values.astype("float32")
y_val   = val_df[TARGET].values.astype("int")
X_val_t = torch.from_numpy(X_val).to(device)
print("device:", device, " 피처:", input_dim, " 양성:", int(y_train.sum()))

C:\Users\Administrator\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: cpu  피처: 42  양성: 9034


## 1. WeightedRandomSampler (불균형 대응)
배치를 뽑을 때 소수 클래스(Yes)를 자주 뽑도록 가중치를 준다.
SMOTE처럼 가짜 데이터를 만들지 않고, 원본 데이터를 더 자주 보여주는 방식이라 더 자연스럽다.

In [2]:
def make_weighted_loader(X, y, batch_size=4096):
    y_int = y.astype(int)
    class_count = np.bincount(y_int)
    sample_w = (1.0 / class_count)[y_int]          # 소수 클래스일수록 큰 가중치
    sampler = WeightedRandomSampler(torch.from_numpy(sample_w).double(),
                                    num_samples=len(y_int), replacement=True)
    ds = TensorDataset(torch.from_numpy(X), torch.from_numpy(y.astype("float32")))
    return DataLoader(ds, batch_size=batch_size, sampler=sampler, drop_last=True)

## 2. 모델: ResidualMLP + Wide&Deep
- 잔차블록(ResBlock): x + MLP(x) 형태로 깊어도 학습이 잘 되게 한다.
- Wide 경로: 원본 피처를 출력에 바로 더해(선형) '단순 기억' 경로를 같이 둔다.
- BatchNorm으로 안정화.

In [3]:
class ResBlock(nn.Module):
    def __init__(self, dim, dropout):
        super().__init__()
        self.lin1 = nn.Linear(dim, dim)
        self.lin2 = nn.Linear(dim, dim)
        self.bn = nn.BatchNorm1d(dim)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        h = torch.relu(self.lin1(x))
        h = self.drop(h)
        h = self.lin2(h)
        return torch.relu(self.bn(x + h))

class ResidualMLP(nn.Module):
    def __init__(self, input_dim, hidden=128, n_blocks=2, dropout=0.2, use_wide=True):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, hidden)
        self.blocks = nn.ModuleList([ResBlock(hidden, dropout) for _ in range(n_blocks)])
        self.head = nn.Linear(hidden, 1)
        self.wide = nn.Linear(input_dim, 1) if use_wide else None
    def forward(self, x):
        h = torch.relu(self.input_proj(x))
        for b in self.blocks:
            h = b(h)
        out = self.head(h).squeeze(1)
        if self.wide is not None:
            out = out + self.wide(x).squeeze(1)
        return out

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0):
        super().__init__(); self.alpha = alpha; self.gamma = gamma
    def forward(self, logits, targets):
        bce = nn.functional.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        p = torch.sigmoid(logits)
        p_t = p * targets + (1 - p) * (1 - targets)
        a_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        return (a_t * (1 - p_t) ** self.gamma * bce).mean()

## 3. 학습 함수 (PR_AUC 기준 early stopping)
매 epoch마다 val PR_AUC를 재서, **PR_AUC가 가장 좋았던 시점의 확률**을 저장한다.
loss가 아니라 우리가 원하는 지표(PR_AUC)로 멈춤/저장한다.

In [4]:
def train_es(model, loader, alpha, gamma, lr, wd, max_epochs=25, patience=5):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    loss_fn = FocalLoss(alpha, gamma)
    best_ap, best_prob, wait = -1.0, None, 0
    for ep in range(max_epochs):
        model.train()
        for xb, yb in loader:
            xb = xb.to(device); yb = yb.to(device)
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            prob = torch.sigmoid(model(X_val_t)).cpu().numpy()
        ap = average_precision_score(y_val, prob)
        if ap > best_ap + 1e-5:
            best_ap, best_prob, wait = ap, prob, 0
        else:
            wait += 1
            if wait >= patience:
                break
    return best_ap, best_prob

## 4. Optuna 탐색 (서브샘플로 빠르게)
CPU에서 전체로 매 trial 돌리면 너무 느리다. 그래서 **탐색은 10만 서브샘플**로 빠르게 하고,
가장 좋은 설정만 뒤에서 전체 데이터로 다시 학습한다.

In [5]:
rng = np.random.default_rng(42)
sub = rng.choice(len(X_train), size=100000, replace=False)
sub_loader = make_weighted_loader(X_train[sub], y_train[sub])

def objective(trial):
    hidden   = trial.suggest_categorical("hidden", [64, 128, 256])
    n_blocks = trial.suggest_int("n_blocks", 1, 3)
    dropout  = trial.suggest_float("dropout", 0.0, 0.5)
    lr       = trial.suggest_float("lr", 1e-4, 3e-3, log=True)
    wd       = trial.suggest_float("wd", 1e-6, 1e-3, log=True)
    alpha    = trial.suggest_float("alpha", 0.5, 0.9)
    gamma    = trial.suggest_float("gamma", 1.0, 3.0)
    use_wide = trial.suggest_categorical("use_wide", [True, False])
    set_seed(42)
    model = ResidualMLP(input_dim, hidden, n_blocks, dropout, use_wide)
    ap, _ = train_es(model, sub_loader, alpha, gamma, lr, wd, max_epochs=10, patience=3)
    return ap

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=12)
print("best PR_AUC(sub):", round(study.best_value, 4))
print("best params:", study.best_params)

best PR_AUC(sub): 0.1654
best params: {'hidden': 128, 'n_blocks': 2, 'dropout': 0.4847685553939328, 'lr': 0.002382900342041922, 'wd': 2.6379257711010443e-05, 'alpha': 0.5154900799844733, 'gamma': 1.0311895993169127, 'use_wide': False}


## 5. 최고 설정으로 전체 데이터 재학습 + 기록
탐색에서 찾은 설정으로 전체 train(135만)에 WeightedRandomSampler + PR_AUC early stopping 적용.

In [6]:
bp = study.best_params
full_loader = make_weighted_loader(X_train, y_train, batch_size=4096)
set_seed(42)
final = ResidualMLP(input_dim, bp["hidden"], bp["n_blocks"], bp["dropout"], bp["use_wide"])
best_ap, best_prob = train_es(final, full_loader, bp["alpha"], bp["gamma"], bp["lr"], bp["wd"],
                              max_epochs=30, patience=6)

m = compute_metrics(y_val, best_prob)
m["notes"] = "ResidualMLP+Wide, WeightedSampler, Focal(Optuna), PR_AUC early stopping"
log_result("MLP_V3", m, path=str(RESULTS_V3))
np.save(NB_DIR / "mlp_v3_val_prob.npy", best_prob)
print("MLP_V3 PR_AUC:", round(m["PR_AUC"], 4))
print("기존 MLP_v2 0.198 대비:", round(m["PR_AUC"] - 0.198, 4))
print("XGB_v2_tuned 0.296 대비:", round(m["PR_AUC"] - 0.296, 4))

MLP_V3 PR_AUC: 0.2305
기존 MLP_v2 0.198 대비: 0.0325
XGB_v2_tuned 0.296 대비: -0.0655


## 6. Threshold sweep (운영점)
PR_AUC는 모델 비교용. 운영점은 threshold를 정한 뒤 F1/F2/Recall/Precision으로 본다.

In [7]:
prob = best_prob
print("t=0.5:", "Recall", round(recall_score(y_val,(prob>=0.5).astype(int)),3),
      "Prec", round(precision_score(y_val,(prob>=0.5).astype(int),zero_division=0),3),
      "F1", round(f1_score(y_val,(prob>=0.5).astype(int),zero_division=0),3))
ts = np.linspace(0.01, 0.99, 99)
f1s = [f1_score(y_val,(prob>=t).astype(int),zero_division=0) for t in ts]
f2s = [fbeta_score(y_val,(prob>=t).astype(int),beta=2,zero_division=0) for t in ts]
for nm, arr in [("F1최적", f1s), ("F2최적", f2s)]:
    t = ts[int(np.argmax(arr))]; pred=(prob>=t).astype(int)
    print(f"{nm}: t={t:.2f} Recall {recall_score(y_val,pred):.3f} "
          f"Prec {precision_score(y_val,pred,zero_division=0):.3f} F1 {f1_score(y_val,pred,zero_division=0):.3f}")

t=0.5: Recall 0.877 Prec 0.084 F1 0.154


F1최적: t=0.88 Recall 0.400 Prec 0.265 F1 0.319
F2최적: t=0.80 Recall 0.658 Prec 0.173 F1 0.274


## 7. 결과 비교 (results_v3_dl.csv)

In [8]:
res = pd.read_csv(RESULTS_V3).drop_duplicates(subset="model", keep="last").sort_values("PR_AUC", ascending=False)
res

,model,PR_AUC,ROC_AUC,Recall,Precision,F1,threshold,notes
0,MLP_V3,0.2305,0.9586,0.8774,0.0843,0.1538,0.5,"ResidualMLP+Wide, WeightedSampler, Focal(Optun..."


---
### 보고서용 쉬운 설명
- **PR_AUC**: 모델이 '백오더 위험이 큰 제품'을 얼마나 앞쪽에 잘 줄 세우는지 보는 지표(불균형에 적합).
- **F1**: threshold를 정한 뒤 Precision(경고 정확도)과 Recall(놓치지 않는 능력)의 균형.
- 그래서 **먼저 PR_AUC로 모델을 고르고, 그다음 threshold를 조정해 운영점(F1/Recall)을 맞춘다.**

### 결론 (실행 후)
- MLP_V3가 기존 MLP 0.198을 넘었나? (구조+sampler+Optuna+PR_AUC ES 효과)
- XGB 0.296과의 격차는 얼마나 남았나? 못 넘으면 정직하게: 정형 데이터에서 DL이 트리에 못 미치는 한계를 기법을 다 써본 뒤 보인 것.